# 23 — CRDT G-Counter (Amazon FAR style)

A grow-only counter that multiple replicas increment independently and merge without coordination or
conflicts — a Conflict-free Replicated Data Type. Parts 1-3 build the core (concurrency at Part 3);
Parts 4-5 are stretch tasks (multi-replica merge properties, then parallel merge-reduce). Fill stubs,
run each test cell, peek at the collapsed `ref_` solutions only after trying.

State is `dict[replica_id -> count]`; the value is the **sum** across replicas; merge is the
**per-replica max**.

---

## Part 1 — G-Counter

`GCounter` with `incr(replica, n=1)`, `value()` (sum over replicas), and `state()` (a copy of the
per-replica dict).

**Lock down:** each replica only ever increases its own entry; the logical value is the sum; the
per-replica breakdown is what makes conflict-free merge possible (Part 2).

In [ ]:
class GCounter:
    def __init__(self):
        raise NotImplementedError

    def incr(self, replica, n=1):
        raise NotImplementedError

    def value(self):
        raise NotImplementedError

    def state(self):
        raise NotImplementedError

In [ ]:
# --- Part 1 tests ---
def _t1():
    c = GCounter()
    c.incr("r1"); c.incr("r1", 5); c.incr("r2", 2)
    assert c.value() == 8
    assert c.state() == {"r1": 6, "r2": 2}
    print("Part 1 OK")

_t1()

## Part 2 — Merge

`merge(a, b) -> dict`: combine two G-Counter states by taking the **maximum** count per replica.

**Lock down:** per-replica max is correct because each replica's entry is monotonically increasing, so
the larger value is the more recent. Merge is **commutative, associative, and idempotent** — the CRDT
properties that make eventual consistency work regardless of message order or duplication.

In [ ]:
def merge(a, b):
    raise NotImplementedError

In [ ]:
# --- Part 2 tests ---
def _t2():
    a = {"r1": 3, "r2": 1}
    b = {"r1": 2, "r2": 5, "r3": 4}
    m = merge(a, b)
    assert m == {"r1": 3, "r2": 5, "r3": 4}            # per-replica max
    assert merge(a, b) == merge(b, a)                   # commutative
    assert merge(m, m) == m                             # idempotent
    assert sum(m.values()) == 12
    print("Part 2 OK")

_t2()

## Part 3 — Concurrent increments

`ConcurrentGCounter`: thread-safe `incr(replica, n=1)`, `value()`, `state()`. Many threads increment
(typically each its **own** replica) at once with no lost updates.

**The invariant:** 8 threads, each incrementing its own replica 1000 times, gives `value() == 8000`
and `state()[ri] == 1000`. **Discuss:** in a real CRDT each replica is a separate *node* (no shared
lock needed — that's the appeal); here we simulate replicas as threads in one process, so we guard the
shared dict. Sharding by replica means threads touch disjoint keys.

In [ ]:
import threading


class ConcurrentGCounter:
    def __init__(self):
        raise NotImplementedError

    def incr(self, replica, n=1):
        raise NotImplementedError

    def value(self):
        raise NotImplementedError

    def state(self):
        raise NotImplementedError

In [ ]:
# --- Part 3 tests ---
def _t3():
    c = ConcurrentGCounter()

    def worker(rid):
        for _ in range(1000):
            c.incr("r%d" % rid)

    ts = [threading.Thread(target=worker, args=(i,)) for i in range(8)]
    for t in ts: t.start()
    for t in ts: t.join()
    assert c.value() == 8000
    assert c.state() == {"r%d" % i: 1000 for i in range(8)}
    print("Part 3 OK")

_t3()

## Part 4 (stretch) — Merge many replicas

`merge_all(states) -> dict`: fold `merge` over a list of replica states (per-replica max across all).
Because merge is commutative/associative/idempotent, the result is **independent of order** and
**unaffected by duplicate** states.

**Discuss:** this is the convergence guarantee of state-based CRDTs (CvRDTs) — replicas gossip states
in any order, with retries/duplicates, and still converge to the same value. The recurring
fault-tolerance theme, but here correctness is *by construction*.

In [ ]:
def merge_all(states):
    raise NotImplementedError

In [ ]:
# --- Part 4 tests ---
def _t4():
    import random
    states = [{"r1": 3}, {"r1": 5, "r2": 2}, {"r2": 4, "r3": 1}, {"r1": 1}]
    m = merge_all(states)
    assert m == {"r1": 5, "r2": 4, "r3": 1}
    shuffled = states[:]; random.Random(0).shuffle(shuffled)
    assert merge_all(shuffled) == m                     # order-independent
    assert merge_all(states + [states[0]]) == m         # duplicates don't change the result
    print("Part 4 OK")

_t4()

## Part 5 (stretch) — Parallel merge-reduce

`parallel_merge(states, num_procs) -> dict`: merge a large pile of replica states across processes with
`ProcessPoolExecutor` — each worker merges a chunk (worker `crdt_workers.merge_states`), then the parent
merges the chunk results.

**Discuss:** merge is associative, so you can reduce in any grouping/parallelism and get the same
answer (a parallel fold); GIL (merging many dicts is CPU-bound -> processes); this is exactly how
CRDT state converges at scale.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import crdt_workers


def parallel_merge(states, num_procs) -> dict:
    raise NotImplementedError

In [ ]:
# --- Part 5 tests ---
def _t5():
    states = [{"r%d" % (i % 5): i} for i in range(50)]
    expected = {}
    for i in range(50):
        r = "r%d" % (i % 5)
        expected[r] = max(expected.get(r, 0), i)
    assert parallel_merge(states, 4) == expected
    print("Part 5 OK")

_t5()

## Likely follow-ups
- PN-Counter (increment + decrement) as two G-Counters; why a single counter can't decrement.
- Other CRDTs: OR-Set, LWW-Register (and the role of version vectors / Lamport clocks).
- State-based (CvRDT) vs operation-based (CmRDT); delta-CRDTs to shrink gossip messages.

---
## Reference solutions (don't peek until you've tried)

In [ ]:
import threading
from concurrent.futures import ProcessPoolExecutor
import crdt_workers


class RefGCounter:
    def __init__(self):
        self.counts = {}

    def incr(self, replica, n=1):
        self.counts[replica] = self.counts.get(replica, 0) + n

    def value(self):
        return sum(self.counts.values())

    def state(self):
        return dict(self.counts)


def ref_merge(a, b):
    out = dict(a)
    for k, v in b.items():
        out[k] = max(out.get(k, 0), v)
    return out


class RefConcurrentGCounter:
    def __init__(self):
        self.counts = {}
        self.lock = threading.Lock()

    def incr(self, replica, n=1):
        with self.lock:
            self.counts[replica] = self.counts.get(replica, 0) + n

    def value(self):
        with self.lock:
            return sum(self.counts.values())

    def state(self):
        with self.lock:
            return dict(self.counts)


def ref_merge_all(states):
    out = {}
    for s in states:
        for k, v in s.items():
            out[k] = max(out.get(k, 0), v)
    return out


def ref_parallel_merge(states, num_procs):
    chunks = [states[i::num_procs] for i in range(num_procs)]
    out = {}
    with ProcessPoolExecutor(max_workers=num_procs) as ex:
        for d in ex.map(crdt_workers.merge_states, chunks):
            for k, v in d.items():
                out[k] = max(out.get(k, 0), v)
    return out


_c = RefGCounter(); _c.incr("a", 2); _c.incr("b"); assert _c.value() == 3 and _c.state() == {"a": 2, "b": 1}
assert ref_merge({"a": 1, "b": 4}, {"a": 3}) == {"a": 3, "b": 4}
assert ref_merge_all([{"a": 1}, {"a": 5}, {"b": 2}]) == {"a": 5, "b": 2}
assert ref_parallel_merge([{"a": 1}, {"a": 9}, {"b": 3}, {"b": 1}], 2) == {"a": 9, "b": 3}
print("reference solutions OK")